In [1]:
import yaml
import numpy as np 
from pytorch_lightning import Trainer
import importlib

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Goal is to test alternate architecte pipelines (eg non feature-based gain models)

In [2]:
path = 'config/attentional_cue/attn_cue_lr_1e-4_bs_64_constrained_slope_multi_distractor.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [3]:
config['data']['loader']['num_workers'] = 0
config['data']['loader']['batch_size'] = 4
config['data']['audio']['rep_kwargs']['rep_on_gpu'] = True
# config['data']['audio']['rep_kwargs']['env_sr'] = 10000
config['data']['fc_size'] = 512
config['concat_arch'] = True

In [4]:
config

{'data': {'num_words': 794,
  'multi_distractor': True,
  'corpus': {'n_talkers': [1, 5],
   'root': '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/'},
  'audio': {'rep_type': 'cochlea_filt',
   'rep_kwargs': {'sr': 20000,
    'env_sr': 8000,
    'n_channels': 40,
    'low_lim': 40,
    'use_pad': True,
    'rep_on_gpu': True,
    'env_extraction_type': 'Half-wave Rectification',
    'downsampling_type': 'TorchTransformsResample',
    'downsampling_kwargs': {'lowpass_filter_width': 64,
     'rolloff': 0.9475937167399596,
     'resampling_method': 'kaiser_window',
     'beta': 14.769656459379492}},
   'compression_type': 'coch_p3',
   'compression_kwargs': {'scale': 1,
    'offset': 1e-07,
    'clip_value': 5,
    'power': 0.3}},
  'loader': {'batch_size': 4, 'num_workers': 0},
  'fc_size': 512},
 'noise_kwargs': {'low_snr': -10, 'high_snr': 10},
 'val_metric': '

In [5]:
import copy

alt_config = copy.deepcopy(config)

In [6]:
alt_config['concat_arch'] = False
alt_config['channel_arch'] = True



In [7]:
from src import attn_tracking_lightning
importlib.reload(attn_tracking_lightning)

concat_module = attn_tracking_lightning.AttentionalTrackingModule(config)
channel_module = attn_tracking_lightning.AttentionalTrackingModule(alt_config)

ln_first
Using concat attention architecture
center_crop=False
binaural=False
using FIR cochleagram
ln_first
Using dual channel architecture
center_crop=False
binaural=False
using FIR cochleagram


In [8]:
trainer = Trainer(
    precision=32,
    # default_root_dir='test_log_dump/',
    # val_check_interval=1000,
#     limit_train_batches=0.,
    limit_val_batches=0.0,
    num_nodes=1,
    gpus=1,
    # detect_anomaly=True,
    accelerator="gpu",
)
# trainer.fit(concat_module)
trainer.fit(channel_module)

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/loops/utilities.py:91: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type                   | Params
-----------------------------------------------------------------
0 | bg_combine_transforms | AudioCompose           | 0     
1 | audio_transforms      | AudioCompose           | 0     
2 | model                 | AttnSequentialAttacker | 66.9 M
3 | loss_fn               | CrossEntropyLoss       | 0     
4 | train_acc             | Accuracy               | 0     
5 | valid_acc             | Accuracy               | 0     
6 | test_acc              | Accuracy               | 0     

<class 'corpus.jsinV3_attn_multi_talker_w_audioset.jsinV3_attn_multi_talker_w_audioset'>


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:240: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 48 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Epoch 0:   0%|          | 11/1452650 [00:09<365:54:06,  1.10it/s, loss=6.82, v_num=3.09e+7]

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/trainer.py:726: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")
